# 00.02 — Matrix Operations & Transformations

**You should already know:** vectors, dot products, signed-number arithmetic, and Cartesian coordinate pairs.  
**By the end, you will be able to:**

1. check whether two matrix shapes can be multiplied and state the output shape;
2. calculate a matrix-vector product as row-by-column dot products;
3. distinguish matrix addition and scalar multiplication from matrix multiplication;
4. apply scaling and rotation matrices to 2-D coordinate columns; and
5. reproduce and verify a transformation calculation with NumPy.

**Difficulty:** ★☆☆☆☆ · **Time:** about 40 minutes

## 1. The Problem

A vector can describe one point or one collection of features. A matrix gives a repeatable rule for turning those inputs into outputs: it can stretch a coordinate, mix features, or act on a whole batch at once.

We will make the rule visible with 2-D geometry first. The same shape logic later appears in NumPy and machine-learning calculations. All coordinates in this lesson are deliberately synthetic teaching values, not observations or evidence about a model's real-world quality.

## 2. A Small Batch of Points

Use fixed Cartesian axes: horizontal \(x\), then vertical \(y\). Coordinates are in metres (m). Each point is a **column**, so the three columns below form a batch:

$$
P = \begin{bmatrix} 1 & 2 & -1 \\ 2 & -1 & 0 \end{bmatrix}\,\mathrm{m}.
$$

The columns are \(\mathbf p_A=(1,2)^\mathsf{T}\,\mathrm{m}\), \(\mathbf p_B=(2,-1)^\mathsf{T}\,\mathrm{m}\), and \(\mathbf p_C=(-1,0)^\mathsf{T}\,\mathrm{m}\). Thus \(P\) has shape \((2,3)\): two coordinate rows and three point columns.

We use three dimensionless matrices:

$$
S=\begin{bmatrix}2&0\\0&\tfrac12\end{bmatrix},\qquad
R=\begin{bmatrix}0&-1\\1&0\end{bmatrix},\qquad
I=\begin{bmatrix}1&0\\0&1\end{bmatrix}.
$$

\(S\) doubles \(x\) and halves \(y\). \(R\) rotates a point 90° counter-clockwise about the origin. \(I\) leaves a point unchanged. Their entries are dimensionless, so they map metre coordinates to metre coordinates. The axes stay fixed: these are active point transformations.

## 3. Intuition Before the Arithmetic

A \(2\times2\) matrix has two rows, so it produces two output coordinates. Each row supplies the recipe for one output coordinate. The number of matrix columns must match the number of input-coordinate rows.

Addition and scalar multiplication work entry-by-entry. They are not sequential transformations. Matrix multiplication is the operation that applies a transformation to a point or combines two transformation rules.

If we scale first and rotate second, write

$$T=RS,\qquad T\mathbf p=R(S\mathbf p).$$

The rightmost matrix touches the point first. Reversing the order creates \(SR\), which is a different rule for this non-uniform scale and rotation.

## 4. Shapes and Matrix Operations

For \(A\in\mathbb R^{m\times n}\) and \(B\in\mathbb R^{n\times p}\),

$$A_{m\times n}B_{n\times p}=C_{m\times p},\qquad c_{ij}=\sum_{k=1}^{n}a_{ik}b_{kj}.$$

The adjacent inner dimensions must match; the outer dimensions give the result shape. Here, \(S_{2\times2}P_{2\times3}\) has shape \((2,3)\).

For same-shaped matrices, \((A+B)_{ij}=a_{ij}+b_{ij}\). For a scalar \(\alpha\), \((\alpha A)_{ij}=\alpha a_{ij}\).

## 5. Manual Walkthrough

Start with \(\mathbf p_A=(1,2)^\mathsf{T}\,\mathrm{m}\). Scaling is a row-by-column calculation:

$$
S\mathbf p_A =
\begin{bmatrix}2&0\\0&\tfrac12\end{bmatrix}
\begin{bmatrix}1\\2\end{bmatrix}\mathrm{m}
=\begin{bmatrix}2(1)+0(2)\\0(1)+\tfrac12(2)\end{bmatrix}\mathrm{m}
=\begin{bmatrix}2\\1\end{bmatrix}\mathrm{m}.
$$

Now rotate that result:

$$
R(S\mathbf p_A)=
\begin{bmatrix}0&-1\\1&0\end{bmatrix}
\begin{bmatrix}2\\1\end{bmatrix}\mathrm{m}
=\begin{bmatrix}-1\\2\end{bmatrix}\mathrm{m}.
$$

The output is in quadrant II, as expected when a 90° counter-clockwise rotation sends \((2,1)^\mathsf{T}\) to \((-1,2)^\mathsf{T}\).

In [ ]:
import numpy as np

# S, R, and I are dimensionless. P stores coordinate columns in metres.
S = np.array([[2.0, 0.0], [0.0, 0.5]], dtype=float)
R = np.array([[0.0, -1.0], [1.0, 0.0]], dtype=float)
I = np.eye(2, dtype=float)
P = np.array([[1.0, 2.0, -1.0], [2.0, -1.0, 0.0]], dtype=float)
p_A = P[:, :1]

assert S.shape == R.shape == I.shape == (2, 2)
assert P.shape == (2, 3)
assert p_A.shape == (2, 1)
print(f"S shape: {S.shape}; P shape: {P.shape}")

In [ ]:
# These operations are entry-wise, not sequential transformations.
print("S + I =\n", S + I)
print("0.5 * S =\n", 0.5 * S)

In [ ]:
# @ performs matrix multiplication for these 2-D arrays.
scaled_points = S @ P
T = R @ S
transformed_points = T @ P

print("S @ p_A =\n", S @ p_A)
print("T = R @ S =\n", T)
print("T @ P =\n", transformed_points)

## 6. Order Changes the Result

The combined scale-then-rotate matrix is

$$RS=\begin{bmatrix}0&-\tfrac12\\2&0\end{bmatrix}.$$

Applied to every point column,

$$(RS)P=\begin{bmatrix}-1&\tfrac12&0\\2&4&-2\end{bmatrix}\mathrm{m}.$$

But rotate-then-scale uses \(SR\), not \(RS\), and gives

$$(SR)P=\begin{bmatrix}-4&2&0\\\tfrac12&1&-\tfrac12\end{bmatrix}\mathrm{m}.$$

In [ ]:
reverse_T = S @ R
reverse_order_points = reverse_T @ P
P_rows = P.T
rowwise_transformed_points = P_rows @ T.T

np.testing.assert_array_equal(S + I, [[3.0, 0.0], [0.0, 1.5]])
np.testing.assert_array_equal(0.5 * S, [[1.0, 0.0], [0.0, 0.25]])
np.testing.assert_array_equal(scaled_points, [[2.0, 4.0, -2.0], [1.0, -0.5, 0.0]])
np.testing.assert_array_equal(T, [[0.0, -0.5], [2.0, 0.0]])
np.testing.assert_array_equal(transformed_points, [[-1.0, 0.5, 0.0], [2.0, 4.0, -2.0]])
np.testing.assert_array_equal(R @ (S @ P), (R @ S) @ P)
np.testing.assert_array_equal(reverse_order_points, [[-4.0, 2.0, 0.0], [0.5, 1.0, -0.5]])
assert not np.array_equal(T, reverse_T)
np.testing.assert_array_equal(rowwise_transformed_points, transformed_points.T)
print("PASS: all fixed calculations and shapes agree.")

## 7. Independent Cross-Check

The matrix calculation should agree with the coordinate rules themselves:

$$S(x,y)^\mathsf{T}=(2x,\tfrac12y)^\mathsf{T},\qquad R(u,v)^\mathsf{T}=(-v,u)^\mathsf{T}.$$

The next cell uses those rules on row-wise point storage. It does not use matrix multiplication to produce the transformed coordinates.

In [ ]:
def scale_then_rotate_by_rules(points):
    """Apply (x, y) -> (2x, y/2) -> (-y, x) to row-wise points."""
    x_coordinates = points[:, 0]
    y_coordinates = points[:, 1]
    scaled = np.column_stack((2.0 * x_coordinates, 0.5 * y_coordinates))
    rotated_after_scale = np.column_stack((-scaled[:, 1], scaled[:, 0]))
    return scaled, rotated_after_scale

scaled_by_rules, transformed_by_rules = scale_then_rotate_by_rules(P_rows)
np.testing.assert_array_equal(scaled_by_rules, scaled_points.T)
np.testing.assert_array_equal(transformed_by_rules, transformed_points.T)
print("PASS: coordinate rules agree with matrix multiplication.")

## 8. Visual Explanation

![Equal-scale Cartesian plot of three synthetic points. Circles mark A (1, 2), B (2, -1), and C (-1, 0) in metres. Dashed arrows lead to square scaled points A (2, 1), B (4, -0.5), and C (-2, 0). Solid arrows then lead to triangular scale-then-rotate points A (-1, 2), B (0.5, 4), and C (0, -2). Marker shapes, direct labels, and arrow styles convey the sequence without relying on colour.](assets/transformation_visual.png)

**Synthetic 2-D points under ordered transformations.** Circles are the input points \(P\); dashed arrows lead to square markers after the non-uniform scale \(S\), which doubles \(x\) and halves \(y\); solid arrows then lead to triangle markers after the 90° counter-clockwise rotation \(R\). Axes have equal scale and are measured in metres. The coordinates are a teaching example, not observed data.

## 9. Common Failure Modes

- **Shape mismatch:** \((2,2)@(3,2)\) is undefined because the adjacent dimensions, 2 and 3, do not match.
- **`*` versus `@`:** for NumPy arrays, `*` is entry-by-entry; use `@` for this lesson's matrix products.
- **Mixed storage conventions:** with points in rows, use `P_rows @ T.T`; with points in columns, use `T @ P`.
- **Wrong composition order:** \(RS\mathbf p\) means \(R(S\mathbf p)\), so \(S\) acts first.
- **Overgeneralising:** a correct calculation on these synthetic points validates only the stated arithmetic and code. It does not establish causal claims or real-world model quality.

## 10. Exercises

1. State whether each expression is valid and, if it is, give its output shape: `T @ P`, `P.T @ T.T`, and `T @ P.T`.
2. Point \(A\) becomes \((-1,2)^\mathsf{T}\,\mathrm{m}\) for one order and \((-4,\tfrac12)^\mathsf{T}\,\mathrm{m}\) for the other. Which result is scale-then-rotate? Explain using parentheses.
3. Complete this equality check for row-wise storage: `np.testing.assert_array_equal(P_rows @ T.T, ________)`.
4. Without multiplying a full batch, describe what \(S\) does to a point on the \(x\)-axis, on the \(y\)-axis, and with a negative \(x\)-coordinate.

::::{dropdown} Solutions
1. Valid: `(2, 2) @ (2, 3) -> (2, 3)` and `(3, 2) @ (2, 2) -> (3, 2)`. Invalid: `(2, 2) @ (3, 2)`.
2. \((-1,2)^\mathsf{T}\,\mathrm{m}\) is \(R(S\mathbf p_A)\), so it is scale-then-rotate.
3. `(T @ P).T`.
4. \(S\) doubles a nonzero \(x\)-axis coordinate, halves a nonzero \(y\)-axis coordinate, and preserves the sign of a negative \(x\)-coordinate.
::::

## 11. References

1. Sheldon Axler, *Linear Algebra Done Right*, 4th edition: https://linear.axler.net/
2. Gilbert Strang, *18.06 Linear Algebra*, MIT OpenCourseWare: https://ocw.mit.edu/courses/18-06-linear-algebra-spring-2010/
3. NumPy Developers, [`numpy.matmul` — NumPy v1.26 Manual](https://numpy.org/doc/1.26/reference/generated/numpy.matmul.html)